# Product Sales Analyzer — VIBE SUSHI

An end-to-end portfolio project covering sales data preparation, product-name mapping, and revenue analysis.

**Analysis period:** June 1–7, 2026. **Currency:** PLN (zł).

## Contents

1. [Environment setup](#setup)
2. [Sales data ingestion](#sales-load)
3. [Sales data preparation](#sales-clean)
4. [Cost and margin reference data](#margin)
5. [Product-name mapping](#mapping)
6. [Data integration](#joins)
7. [Data checks and estimated costs](#checks)
8. [Revenue analysis](#analysis)
9. [Findings and next steps](#next)

## Data definitions

| Field | Definition |
|---|---|
| `date` | Daily export date; not an individual order identifier |
| `name` | Product name in the sales export |
| `qty` | Units of the listed product sold; a bundle counts as one unit, not its individual components |
| `sum` | Revenue for the row; the original Choice export labels this field “Цена” |
| `price` | Revenue divided by quantity; after aggregation, the quantity-weighted average selling price |
| `unit_cost` | Unit product cost from the available reference table |
| `total_cost` | Estimated product cost for the quantity sold: `unit_cost × qty` |
| `unit_profit` | Reference profit based on website pricing; not realized profit for June sales |
| `marginal_profit`, `food_cost` | Reference percentage metrics imported from the margin table |
| `revenue_share_pct` | Product revenue as a percentage of total revenue within the analysis scope |

## Scope and limitations

- Rows classified as add-ons (“Дополнение”) are excluded. Reported revenue covers the retained product rows and does not necessarily equal total restaurant revenue.
- The export does not identify the sales channel, and website prices changed over time. Channels are not inferred from prices; realized profit after marketplace commissions is outside scope.
- Reference costs may differ from historical costs during June. Cost-based figures are estimates.
- Product mappings were reviewed by the project author. Selected similar products use accepted cost proxies; a mapped name does not guarantee an identical recipe.
- Findings describe one week and should not be generalized to the entire month.


<a id="setup"></a>
## 1. Environment setup

Import pandas and path utilities. Relative data paths assume the notebook is run from the project's `notebooks` directory.

In [728]:
import pandas as pd
from pathlib import Path

<a id="sales-load"></a>
## 2. Sales data ingestion

### 2.1. Load daily exports

Read `data/raw/2026.06.01.csv` through `2026.06.07.csv`. Each file represents one day of sales.

In [729]:
df_01_06 = pd.read_csv(Path.cwd().parent / 'data' / 'raw' / '2026.06.01.csv')
df_02_06 = pd.read_csv(Path.cwd().parent / 'data' / 'raw' / '2026.06.02.csv')
df_03_06 = pd.read_csv(Path.cwd().parent / 'data' / 'raw' / '2026.06.03.csv')
df_04_06 = pd.read_csv(Path.cwd().parent / 'data' / 'raw' / '2026.06.04.csv')
df_05_06 = pd.read_csv(Path.cwd().parent / 'data' / 'raw' / '2026.06.05.csv')
df_06_06 = pd.read_csv(Path.cwd().parent / 'data' / 'raw' / '2026.06.06.csv')
df_07_06 = pd.read_csv(Path.cwd().parent / 'data' / 'raw' / '2026.06.07.csv')

### 2.2. Assign reporting dates

Attach the corresponding date to each daily export and inspect the source data types.

In [730]:
df_01_06['Период'] = '01.06.2026'
df_02_06['Период'] = '02.06.2026'
df_03_06['Период'] = '03.06.2026'
df_04_06['Период'] = '04.06.2026'
df_05_06['Период'] = '05.06.2026'
df_06_06['Период'] = '06.06.2026'
df_07_06['Период'] = '07.06.2026'

In [731]:
df_01_06.dtypes

Тип                     str
Название позиции        str
Продано               int64
Цена                float64
Период                  str
Location                str
dtype: object

### 2.3. Combine daily tables

Concatenate the daily datasets into `sales` and inspect the combined source structure.

In [732]:
dfs = [df_01_06, df_02_06, df_03_06, df_04_06, df_05_06, df_06_06, df_07_06]

In [733]:
sales = pd.concat(dfs, ignore_index=True)

In [734]:
sales

,Тип,Название позиции,Продано,Цена,Период,Location
0,Позиция,Futomaki ŁOSOŚ GRILLOWANY | 6 szt,6,228.0,01.06.2026,VIBE SUSHI
1,Позиция,Nigiri łosoś | 2 szt,4,100.0,01.06.2026,VIBE SUSHI
2,Позиция,PROMO 1+1=3 Sushi burgery,3,360.0,01.06.2026,VIBE SUSHI
3,Позиция,Hosomaki ŁOSOŚ SUROWY | 8 szt,3,75.0,01.06.2026,VIBE SUSHI
4,Позиция,Futomaki TUŃCZYK SUROWY | 6 szt,2,90.0,01.06.2026,VIBE SUSHI
...,...,...,...,...,...,...
186,Дополнение,Imbir,2,3.0,07.06.2026,VIBE SUSHI
187,Дополнение,Wasabi,2,4.0,07.06.2026,VIBE SUSHI
188,Дополнение,Sos teriyaki 30ml,1,3.0,07.06.2026,VIBE SUSHI
189,Дополнение,Krabs - Burger,1,25.0,07.06.2026,VIBE SUSHI


<a id="sales-clean"></a>
## 3. Sales data preparation

### 3.1. Standardize column names

Rename source fields consistently. The source field “Цена” contains row revenue and is therefore renamed `sum`.

In [735]:
sales = sales.rename(columns={
    'Тип': 'type',
    'Название позиции': 'name',
    'Продано': 'qty',
    'Цена': 'sum',
    'Период': 'date',
    'Location': 'location'
    })

In [736]:
sales

,type,name,qty,sum,date,location
0,Позиция,Futomaki ŁOSOŚ GRILLOWANY | 6 szt,6,228.0,01.06.2026,VIBE SUSHI
1,Позиция,Nigiri łosoś | 2 szt,4,100.0,01.06.2026,VIBE SUSHI
2,Позиция,PROMO 1+1=3 Sushi burgery,3,360.0,01.06.2026,VIBE SUSHI
3,Позиция,Hosomaki ŁOSOŚ SUROWY | 8 szt,3,75.0,01.06.2026,VIBE SUSHI
4,Позиция,Futomaki TUŃCZYK SUROWY | 6 szt,2,90.0,01.06.2026,VIBE SUSHI
...,...,...,...,...,...,...
186,Дополнение,Imbir,2,3.0,07.06.2026,VIBE SUSHI
187,Дополнение,Wasabi,2,4.0,07.06.2026,VIBE SUSHI
188,Дополнение,Sos teriyaki 30ml,1,3.0,07.06.2026,VIBE SUSHI
189,Дополнение,Krabs - Burger,1,25.0,07.06.2026,VIBE SUSHI


### 3.2. Derive unit prices and parse dates

Calculate `price = sum / qty`, convert reporting dates to datetime, and inspect the resulting types.

In [737]:
sales['price'] = sales['sum'] / sales['qty']

In [738]:
sales['date'] = pd.to_datetime(sales['date'], format='%d.%m.%Y')

In [739]:
sales.dtypes

type                   str
name                   str
qty                  int64
sum                float64
date        datetime64[us]
location               str
price              float64
dtype: object

In [740]:
sales

,type,name,qty,sum,date,location,price
0,Позиция,Futomaki ŁOSOŚ GRILLOWANY | 6 szt,6,228.0,2026-06-01,VIBE SUSHI,38.0
1,Позиция,Nigiri łosoś | 2 szt,4,100.0,2026-06-01,VIBE SUSHI,25.0
2,Позиция,PROMO 1+1=3 Sushi burgery,3,360.0,2026-06-01,VIBE SUSHI,120.0
3,Позиция,Hosomaki ŁOSOŚ SUROWY | 8 szt,3,75.0,2026-06-01,VIBE SUSHI,25.0
4,Позиция,Futomaki TUŃCZYK SUROWY | 6 szt,2,90.0,2026-06-01,VIBE SUSHI,45.0
...,...,...,...,...,...,...,...
186,Дополнение,Imbir,2,3.0,2026-06-07,VIBE SUSHI,1.5
187,Дополнение,Wasabi,2,4.0,2026-06-07,VIBE SUSHI,2.0
188,Дополнение,Sos teriyaki 30ml,1,3.0,2026-06-07,VIBE SUSHI,3.0
189,Дополнение,Krabs - Burger,1,25.0,2026-06-07,VIBE SUSHI,25.0


### 3.3. Remove the location field

The current analysis covers a single location, so the location column is removed from the working table.

In [741]:
sales = sales.drop(labels=['location'], axis=1)

In [742]:
sales

,type,name,qty,sum,date,price
0,Позиция,Futomaki ŁOSOŚ GRILLOWANY | 6 szt,6,228.0,2026-06-01,38.0
1,Позиция,Nigiri łosoś | 2 szt,4,100.0,2026-06-01,25.0
2,Позиция,PROMO 1+1=3 Sushi burgery,3,360.0,2026-06-01,120.0
3,Позиция,Hosomaki ŁOSOŚ SUROWY | 8 szt,3,75.0,2026-06-01,25.0
4,Позиция,Futomaki TUŃCZYK SUROWY | 6 szt,2,90.0,2026-06-01,45.0
...,...,...,...,...,...,...
186,Дополнение,Imbir,2,3.0,2026-06-07,1.5
187,Дополнение,Wasabi,2,4.0,2026-06-07,2.0
188,Дополнение,Sos teriyaki 30ml,1,3.0,2026-06-07,3.0
189,Дополнение,Krabs - Burger,1,25.0,2026-06-07,25.0


### 3.4. Define the product-sales scope

Exclude rows classified as add-ons (“Дополнение”). All subsequent revenue totals refer to the retained rows.

In [743]:
sales = sales[sales['type'] != 'Дополнение']

In [744]:
sales

,type,name,qty,sum,date,price
0,Позиция,Futomaki ŁOSOŚ GRILLOWANY | 6 szt,6,228.0,2026-06-01,38.0
1,Позиция,Nigiri łosoś | 2 szt,4,100.0,2026-06-01,25.0
2,Позиция,PROMO 1+1=3 Sushi burgery,3,360.0,2026-06-01,120.0
3,Позиция,Hosomaki ŁOSOŚ SUROWY | 8 szt,3,75.0,2026-06-01,25.0
4,Позиция,Futomaki TUŃCZYK SUROWY | 6 szt,2,90.0,2026-06-01,45.0
...,...,...,...,...,...,...
179,Позиция,Miso z łososiem | 350 ml,1,35.0,2026-06-07,35.0
180,Позиция,Futomaki FIRE HARMONY | 6 szt,1,58.0,2026-06-07,58.0
181,Позиция,Hosomaki AWOKADO | 8 szt,1,21.0,2026-06-07,21.0
182,Позиция,Futomaki ŁOSOŚ SUROWY | 6 szt,1,38.0,2026-06-07,38.0


### 3.5. Review the prepared sales table

Retain date, product name, quantity, unit price, and revenue. Inspect data types and unique product names before mapping.

In [745]:
sales = sales[['date', 'name', 'qty', 'price', 'sum']]

In [746]:
sales

,date,name,qty,price,sum
0,2026-06-01,Futomaki ŁOSOŚ GRILLOWANY | 6 szt,6,38.0,228.0
1,2026-06-01,Nigiri łosoś | 2 szt,4,25.0,100.0
2,2026-06-01,PROMO 1+1=3 Sushi burgery,3,120.0,360.0
3,2026-06-01,Hosomaki ŁOSOŚ SUROWY | 8 szt,3,25.0,75.0
4,2026-06-01,Futomaki TUŃCZYK SUROWY | 6 szt,2,45.0,90.0
...,...,...,...,...,...
179,2026-06-07,Miso z łososiem | 350 ml,1,35.0,35.0
180,2026-06-07,Futomaki FIRE HARMONY | 6 szt,1,58.0,58.0
181,2026-06-07,Hosomaki AWOKADO | 8 szt,1,21.0,21.0
182,2026-06-07,Futomaki ŁOSOŚ SUROWY | 6 szt,1,38.0,38.0


In [747]:
sales.dtypes

date     datetime64[us]
name                str
qty               int64
price           float64
sum             float64
dtype: object

In [748]:
sorted(sales['name'].unique())

['Amour Set | 36 szt',
 'Atlantic Wave roll | 8 szt',
 'Black dragon roll | 8 szt',
 'California KRAB-MIX | 8 szt',
 'California KREWETKA GOTOWANA | 8 szt',
 'California RAINBOW | 8 szt',
 'California TUŃCZYK SUROWY | 8 szt',
 'California z czarnym ryżem ŁOSOŚ SUROWY | 8 szt',
 'Cheese philadelphia | 8 szt',
 'Duo set | 28 szt',
 'Fanta jagoda 0.33l',
 'Feliks roll | 8 szt',
 'Fuji roll | 8 szt',
 'Futomaki AWOKADO | 6 szt',
 'Futomaki FIRE HARMONY | 6 szt',
 'Futomaki FRESH ROLL | 6 szt',
 'Futomaki KALMAR PANKO | 6 szt',
 'Futomaki KRAB W TEMPURZE | 6 szt',
 'Futomaki KREWETKA EBI | 6 szt / 12 szt',
 'Futomaki KREWETKA PANKO | 6 szt / 12 szt',
 'Futomaki TUŃCZYK SUROWY | 6 szt',
 'Futomaki TUŃCZYK SUROWY | 6 szt / 12 szt',
 'Futomaki WARZYWA W TEMPURZE | 6 szt',
 'Futomaki ŁOSOŚ GRILLOWANY | 6 szt',
 'Futomaki ŁOSOŚ GRILLOWANY | 6 szt / 12 szt',
 'Futomaki ŁOSOŚ SUROWY | 6 szt',
 'Futomaki ŁOSOŚ SUROWY | 6 szt / 12 szt',
 'Green dragon roll | 8 szt',
 'Grill set | 30 szt',
 'Gunkan E

<a id="margin"></a>
## 4. Cost and margin reference data

### 4.1. Load the source reference table

Read `data/raw/margin_table.csv` and inspect the available fields.

In [749]:
margin = pd.read_csv(Path.cwd().parent / 'data' / 'raw' / 'margin_table.csv')
margin.head(10)

,ПОЗИЦИЯ,КАТЕГОРИЯ,СОСТОЯНИЕ,РЕСТ,РАСХОДЫ,VAT ПРОЦ,ОФЕР 25%,НОВАЯ ЦЕНА,ЦЕНА БРУТТО,ЦЕНА ОДБЮР,...,FC,ОФЕР_Д,БРУТТО_Д,ПРОВИЗИЯ_Д,VAT 8%_Д,ПОЛУЧАЕМ ОТ НИХ_Д,ЧИСТЯК_Д,ПРОФИТ_Д,МАРЖА_Д,FC_Д
0,Miso krewetki,ZUPY,Активно,Vibe,5.45,23.0,26.79,22.0,27.0,24.3,...,29.1%,35.39,40.0,16.00,3.20,24.00,20.80,15.35,73.82%,26.18%
1,Miso łosoś,ZUPY,Активно,Vibe,3.59,8.0,15.50,NaN,27.0,24.3,...,16.0%,23.32,30.0,12.00,2.40,18.00,15.60,12.01,77.00%,23.00%
2,Miso inari tofu,ZUPY,Активно,Vibe,1.51,8.0,6.52,NaN,25.0,22.5,...,7.3%,9.81,11.0,4.40,0.88,6.60,5.72,4.21,73.62%,26.38%
3,Promo set | 10 szt PROMO,ZESTAWY,Активно,Rice,14.93,8.0,45.94,NaN,0.0,0.0,...,NaN,97.02,70.0,28.00,5.60,42.00,36.40,21.47,58.99%,41.01%
4,NESUROWE set | 10 szt PROMO,ZESTAWY,Активно,Rice,15.27,8.0,46.99,NaN,0.0,0.0,...,NaN,99.24,70.0,28.00,5.60,42.00,36.40,21.13,58.06%,41.94%
5,Zestaw Phila | 16 szt,ZESTAWY,Активно,Rice,18.21,8.0,56.05,65.0,64.6,64.6,...,30.6%,118.37,89.6,35.84,7.17,53.76,46.59,28.38,60.91%,39.09%
6,Studencki | 32 szt,ZESTAWY,Активно,Rice,18.90,8.0,58.18,68.0,65.6,65.6,...,31.3%,122.87,91.6,36.64,7.33,54.96,47.63,28.73,60.31%,39.69%
7,Zestaw Hosomaki | 40 szt,ZESTAWY,Активно,Rice,20.57,8.0,63.32,74.0,74.6,74.6,...,30.0%,133.71,103.6,41.44,8.29,62.16,53.87,33.30,61.82%,38.18%
8,Akira | 24 szt,ZESTAWY,Активно,Rice,24.99,8.0,76.93,90.0,89.6,89.6,...,30.3%,162.45,124.6,49.84,9.97,74.76,64.79,39.80,61.43%,38.57%
9,Tokio | 28 szt,ZESTAWY,Активно,Rice,25.50,8.0,78.49,92.0,91.6,91.6,...,30.3%,165.76,127.6,51.04,10.21,76.56,66.35,40.85,61.57%,38.43%


In [750]:
margin.columns

Index(['ПОЗИЦИЯ', 'КАТЕГОРИЯ', 'СОСТОЯНИЕ', 'РЕСТ', 'РАСХОДЫ', 'VAT ПРОЦ',
       'ОФЕР 25%', 'НОВАЯ ЦЕНА', 'ЦЕНА БРУТТО', 'ЦЕНА ОДБЮР', 'VAT',
       'ЦЕНА НЕТТО', 'ПРОФИТ', 'МАРЖА', 'FC', 'ОФЕР_Д', 'БРУТТО_Д',
       'ПРОВИЗИЯ_Д', 'VAT 8%_Д', 'ПОЛУЧАЕМ ОТ НИХ_Д', 'ЧИСТЯК_Д', 'ПРОФИТ_Д',
       'МАРЖА_Д', 'FC_Д'],
      dtype='str')

### 4.2. Select the restaurant and relevant metrics

Filter the reference data to Vibe and retain product attributes, costs, reference prices, and margin metrics.

In [751]:
margin = margin[margin['РЕСТ'] == 'Vibe']

In [752]:
margin = margin[['ПОЗИЦИЯ', 'КАТЕГОРИЯ', 'РАСХОДЫ', 'ЦЕНА БРУТТО', 'ПРОФИТ', 'МАРЖА', 'FC', 'VAT', 'VAT ПРОЦ']]

In [753]:
margin

,ПОЗИЦИЯ,КАТЕГОРИЯ,РАСХОДЫ,ЦЕНА БРУТТО,ПРОФИТ,МАРЖА,FC,VAT,VAT ПРОЦ
0,Miso krewetki,ZUPY,5.45,27.0,13.27,70.90%,29.1%,5.5890,23.0
1,Miso łosoś,ZUPY,3.59,27.0,18.77,83.95%,16.0%,1.9440,8.0
2,Miso inari tofu,ZUPY,1.51,25.0,19.19,92.71%,7.3%,1.8000,8.0
20,Zestaw startowy | 16 szt,ZESTAWY,20.13,81.9,47.68,70.31%,29.7%,5.8968,8.0
21,Hosomaki set | 56 szt,ZESTAWY,23.48,100.0,59.32,71.64%,28.4%,7.2000,8.0
...,...,...,...,...,...,...,...,...,...
215,Shrimp Bowl,BOWL,14.75,50.0,19.90,57.44%,42.6%,10.3500,23.0
216,Crunch tempura Bowl,BOWL,14.99,52.0,21.05,58.40%,41.6%,10.7640,23.0
217,Phila KRAB-MIX,PHILADELPHIA,12.66,55.0,32.88,72.21%,27.8%,3.9600,8.0
218,Miso ramen,RAMEN,12.71,45.0,24.55,65.89%,34.1%,3.2400,8.0


### 4.3. Standardize reference fields

Rename columns and inspect product names. `price_gross` is a reference price that may differ from the price recorded in June sales.

In [754]:
margin = margin.rename(columns={
    'ПОЗИЦИЯ': 'name',
    'КАТЕГОРИЯ': 'category',
    'РАСХОДЫ': 'unit_cost',
    'ЦЕНА БРУТТО': 'price_gross',
    'ПРОФИТ': 'unit_profit',
    'МАРЖА': 'marginal_profit',
    'FC': 'food_cost',
    'VAT': 'vat_as_cost',
    'VAT ПРОЦ': 'vat_rate'
})

In [755]:
margin.sort_values(by='name', axis=0)

,name,category,unit_cost,price_gross,unit_profit,marginal_profit,food_cost,vat_as_cost,vat_rate
111,Akami Tamago roll,ROLKI AUTORSKIE,13.43,50.0,27.97,67.55%,32.5%,3.6000,8.0
104,Atlantic Wave,ROLKI AUTORSKIE,13.87,60.0,27.71,66.65%,33.4%,12.4200,23.0
116,Awokado delux,ROLKI AUTORSKIE,12.37,55.0,33.17,72.84%,27.2%,3.9600,8.0
38,Awokado fresh roll,WEGE MENU,9.53,38.0,21.93,69.71%,30.3%,2.7360,8.0
60,B. Wege,SUSHI BURGER,8.54,50.0,32.86,79.38%,20.6%,3.6000,8.0
...,...,...,...,...,...,...,...,...,...
74,ZR Krewetki,ROLKI ZAPIECZONE,13.31,60.0,28.27,67.99%,32.0%,12.4200,23.0
77,ZR Surowy,ROLKI ZAPIECZONE,12.11,60.0,37.57,75.62%,24.4%,4.3200,8.0
79,ZR Łosoś pieczony,ROLKI ZAPIECZONE,10.58,60.0,39.10,78.70%,21.3%,4.3200,8.0
27,Zestaw Amour | 36 szt,ZESTAWY,42.75,195.0,118.71,73.52%,26.5%,14.0400,8.0


In [756]:
sorted(margin['name'].unique())

['Akami Tamago roll',
 'Atlantic Wave',
 'Awokado delux',
 'Awokado fresh roll',
 'B. Wege',
 'Black Philadelphia',
 'Black dragon roll',
 'Bonito roll',
 'Burger czarny ryż KREWETKA GOTOWANA',
 'Burger grill łosoś',
 'Burger krewetka panko',
 'Burger łosoś surowy',
 'C Black Surowy',
 'Calif LOSOS I SEZAM',
 'Calif krewetka 16/20',
 'California GRILL',
 'California KRAB',
 'California KREW PANKO',
 'California RAINBOW',
 'California SHIITAKE',
 'California SUROWY',
 'California TUNA',
 'California Warzywa w tempurze',
 'Canada roll',
 'Cheddar roll',
 'Crunch tempura Bowl',
 'Duo set | 28 szt',
 'Feliks łosoś',
 'Fibonacci roll',
 'Fuji roll',
 'Futo AWOKADO',
 'Futo EBI',
 'Futo FIRE HARMONY',
 'Futo FRESH ROLL',
 'Futo GRILL',
 'Futo KALMAR TEMPURA',
 'Futo KRAB TEMPURA',
 'Futo KREWETKA TEMPURA',
 'Futo MANGO',
 'Futo SHIITAKE',
 'Futo SUROWY',
 'Futo TATAR',
 'Futo TUNA',
 'Futo WARZYWA TEMPURA',
 'Futo ŁOSOŚ TEMPURA',
 'Futomaki set | 48 szt',
 'Green dragon roll',
 'Green fresh 

<a id="mapping"></a>
## 5. Product-name mapping

### 5.1. Create the mapping template

Build an empty `mapping_df` from distinct sales names. This documents the initial preparation step; the completed mapping is loaded separately below. `match_type` and `reviewed` belong to the template and are not used by the current joins.

In [757]:
mapping_df = pd.DataFrame({'sales_name': sorted(sales['name'].unique())})

In [758]:
mapping_df

,sales_name
0,Amour Set | 36 szt
1,Atlantic Wave roll | 8 szt
2,Black dragon roll | 8 szt
3,California KRAB-MIX | 8 szt
4,California KREWETKA GOTOWANA | 8 szt
...,...
85,Zestaw Philadelphia + Krabs Burger -50%
86,Zestaw Philadelphia | 24 szt
87,Zestaw Sakura | 62 szt
88,Zestaw Startowy | 16 szt


### 5.2. Initial reference exports

Export commands are intentionally commented out. Re-exporting the empty template to the existing mapping path would overwrite reviewed mappings. The processed cost reference also includes manual additions that are not reproduced by this source-preparation block.

In [759]:
# mapping_df.to_csv(Path.cwd().parent / 'data' / 'mappings' / 'product_name_mapping.csv', index=False)

In [760]:
margin.to_csv(Path.cwd().parent / 'data' / 'mappings' / 'margin_table.csv', index=False)

### 5.3. Inspect the prepared inputs

Review the sales and reference tables before loading the completed mappings.

In [761]:
margin

,name,category,unit_cost,price_gross,unit_profit,marginal_profit,food_cost,vat_as_cost,vat_rate
0,Miso krewetki,ZUPY,5.45,27.0,13.27,70.90%,29.1%,5.5890,23.0
1,Miso łosoś,ZUPY,3.59,27.0,18.77,83.95%,16.0%,1.9440,8.0
2,Miso inari tofu,ZUPY,1.51,25.0,19.19,92.71%,7.3%,1.8000,8.0
20,Zestaw startowy | 16 szt,ZESTAWY,20.13,81.9,47.68,70.31%,29.7%,5.8968,8.0
21,Hosomaki set | 56 szt,ZESTAWY,23.48,100.0,59.32,71.64%,28.4%,7.2000,8.0
...,...,...,...,...,...,...,...,...,...
215,Shrimp Bowl,BOWL,14.75,50.0,19.90,57.44%,42.6%,10.3500,23.0
216,Crunch tempura Bowl,BOWL,14.99,52.0,21.05,58.40%,41.6%,10.7640,23.0
217,Phila KRAB-MIX,PHILADELPHIA,12.66,55.0,32.88,72.21%,27.8%,3.9600,8.0
218,Miso ramen,RAMEN,12.71,45.0,24.55,65.89%,34.1%,3.2400,8.0


In [762]:
sales

,date,name,qty,price,sum
0,2026-06-01,Futomaki ŁOSOŚ GRILLOWANY | 6 szt,6,38.0,228.0
1,2026-06-01,Nigiri łosoś | 2 szt,4,25.0,100.0
2,2026-06-01,PROMO 1+1=3 Sushi burgery,3,120.0,360.0
3,2026-06-01,Hosomaki ŁOSOŚ SUROWY | 8 szt,3,25.0,75.0
4,2026-06-01,Futomaki TUŃCZYK SUROWY | 6 szt,2,45.0,90.0
...,...,...,...,...,...
179,2026-06-07,Miso z łososiem | 350 ml,1,35.0,35.0
180,2026-06-07,Futomaki FIRE HARMONY | 6 szt,1,58.0,58.0
181,2026-06-07,Hosomaki AWOKADO | 8 szt,1,21.0,21.0
182,2026-06-07,Futomaki ŁOSOŚ SUROWY | 6 szt,1,38.0,38.0


### 5.4. Load maintained reference files

Read `data/mappings/product_name_mapping.csv` and `data/mappings/margin_table.csv`. The pair `sales_name → margin_name` links product names across the two sources.

In [763]:
mapping = pd.read_csv(Path().cwd().parent / 'data' / 'mappings' / 'product_name_mapping.csv')
mapping = mapping[['sales_name', 'margin_name']]
mapping

,sales_name,margin_name
0,Amour Set | 36 szt,Zestaw Amour | 36 szt
1,Atlantic Wave roll | 8 szt,Atlantic Wave
2,Black dragon roll | 8 szt,Black dragon roll
3,California KRAB-MIX | 8 szt,California KRAB
4,California KREWETKA GOTOWANA | 8 szt,Calif krewetka 16/20
...,...,...
85,Zestaw Philadelphia + Krabs Burger -50%,Zestaw Philadelphia + Krabs Burger -50%
86,Zestaw Philadelphia | 24 szt,Philadelphia | 24 szt
87,Zestaw Sakura | 62 szt,Sakura set | 62 szt
88,Zestaw Startowy | 16 szt,Zestaw startowy | 16 szt


In [764]:
margin_table = pd.read_csv(Path().cwd().parent / 'data' / 'mappings' / 'margin_table.csv')

In [765]:
margin_table

,name,category,unit_cost,price_gross,unit_profit,marginal_profit,food_cost,vat_as_cost,vat_rate
0,Miso krewetki,ZUPY,5.45,27.0,13.27,70.90%,29.1%,5.5890,23.0
1,Miso łosoś,ZUPY,3.59,27.0,18.77,83.95%,16.0%,1.9440,8.0
2,Miso inari tofu,ZUPY,1.51,25.0,19.19,92.71%,7.3%,1.8000,8.0
3,Zestaw startowy | 16 szt,ZESTAWY,20.13,81.9,47.68,70.31%,29.7%,5.8968,8.0
4,Hosomaki set | 56 szt,ZESTAWY,23.48,100.0,59.32,71.64%,28.4%,7.2000,8.0
...,...,...,...,...,...,...,...,...,...
125,Shrimp Bowl,BOWL,14.75,50.0,19.90,57.44%,42.6%,10.3500,23.0
126,Crunch tempura Bowl,BOWL,14.99,52.0,21.05,58.40%,41.6%,10.7640,23.0
127,Phila KRAB-MIX,PHILADELPHIA,12.66,55.0,32.88,72.21%,27.8%,3.9600,8.0
128,Miso ramen,RAMEN,12.71,45.0,24.55,65.89%,34.1%,3.2400,8.0


### 5.5. Retain populated mappings

Select rows with a nonempty `margin_name`. This checks mapping completeness, not semantic correctness.

In [766]:
mapping_matched = mapping[mapping['margin_name'].notna() & mapping['margin_name'].ne('')].copy()

In [767]:
mapping_matched

,sales_name,margin_name
0,Amour Set | 36 szt,Zestaw Amour | 36 szt
1,Atlantic Wave roll | 8 szt,Atlantic Wave
2,Black dragon roll | 8 szt,Black dragon roll
3,California KRAB-MIX | 8 szt,California KRAB
4,California KREWETKA GOTOWANA | 8 szt,Calif krewetka 16/20
...,...,...
85,Zestaw Philadelphia + Krabs Burger -50%,Zestaw Philadelphia + Krabs Burger -50%
86,Zestaw Philadelphia | 24 szt,Philadelphia | 24 szt
87,Zestaw Sakura | 62 szt,Sakura set | 62 szt
88,Zestaw Startowy | 16 szt,Zestaw startowy | 16 szt


<a id="joins"></a>
## 6. Data integration

### 6.1. Attach reference metrics to sales names

Data flow: **sales → mapping → margin_table**. First, join the mapping to the reference table. `validate="many_to_one"` checks that keys in the right-hand table are unique. The following rename handles the legacy cost-column spelling `excepnses`.

In [768]:
margin_by_sales_name = mapping_matched.merge(
    right=margin_table.rename(columns={'name': 'margin_table_name'}),
    left_on='margin_name',
    right_on='margin_table_name',
    how='left',
    validate='many_to_one'
)

In [769]:
margin_by_sales_name = margin_by_sales_name.rename({'excepnses': 'unit_cost'}, axis=1)

In [770]:
margin_by_sales_name

,sales_name,margin_name,margin_table_name,category,unit_cost,price_gross,unit_profit,marginal_profit,food_cost,vat_as_cost,vat_rate
0,Amour Set | 36 szt,Zestaw Amour | 36 szt,Zestaw Amour | 36 szt,ZESTAWY,42.75,195.0,118.71,73.52%,26.5%,14.0400,8.0
1,Atlantic Wave roll | 8 szt,Atlantic Wave,Atlantic Wave,ROLKI AUTORSKIE,13.87,60.0,27.71,66.65%,33.4%,12.4200,23.0
2,Black dragon roll | 8 szt,Black dragon roll,Black dragon roll,ROLKI AUTORSKIE,18.29,70.0,39.67,68.44%,31.6%,5.0400,8.0
3,California KRAB-MIX | 8 szt,California KRAB,California KRAB,CALIFORNIA,7.71,45.0,29.55,79.30%,20.7%,3.2400,8.0
4,California KREWETKA GOTOWANA | 8 szt,Calif krewetka 16/20,Calif krewetka 16/20,CALIFORNIA,15.05,60.0,26.53,63.80%,36.2%,12.4200,23.0
...,...,...,...,...,...,...,...,...,...,...,...
75,Zestaw Philadelphia + Krabs Burger -50%,Zestaw Philadelphia + Krabs Burger -50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
76,Zestaw Philadelphia | 24 szt,Philadelphia | 24 szt,Philadelphia | 24 szt,ZESTAWY,38.04,170.0,102.72,72.98%,27.0%,12.2400,8.0
77,Zestaw Sakura | 62 szt,Sakura set | 62 szt,Sakura set | 62 szt,ZESTAWY,69.78,295.0,174.48,71.43%,28.6%,21.2400,8.0
78,Zestaw Startowy | 16 szt,Zestaw startowy | 16 szt,Zestaw startowy | 16 szt,ZESTAWY,20.13,81.9,47.68,70.31%,29.7%,5.8968,8.0


### 6.2. Enrich sales records

Select the reference metrics and left-join them to `sales`. Unmatched sales remain in the dataset with missing reference values.

In [771]:
columns_to_add = [
    'sales_name',
    'unit_profit',
    'unit_cost',
    'marginal_profit',
    'food_cost',
    'vat_as_cost',
    'vat_rate',
]

In [772]:
sales_final = sales.merge(
    right=margin_by_sales_name[columns_to_add],
    left_on='name',
    right_on='sales_name',
    how='left',
    validate='many_to_one'
)

In [773]:
sales_final

,date,name,qty,price,sum,sales_name,unit_profit,unit_cost,marginal_profit,food_cost,vat_as_cost,vat_rate
0,2026-06-01,Futomaki ŁOSOŚ GRILLOWANY | 6 szt,6,38.0,228.0,Futomaki ŁOSOŚ GRILLOWANY | 6 szt,21.47,7.51,74.10%,25.9%,2.520,8.0
1,2026-06-01,Nigiri łosoś | 2 szt,4,25.0,100.0,Nigiri łosoś | 2 szt,15.88,5.65,73.77%,26.2%,1.872,8.0
2,2026-06-01,PROMO 1+1=3 Sushi burgery,3,120.0,360.0,PROMO 1+1=3 Sushi burgery,64.00,35.36,64.42%,35.6%,8.640,8.0
3,2026-06-01,Hosomaki ŁOSOŚ SUROWY | 8 szt,3,25.0,75.0,Hosomaki ŁOSOŚ SUROWY | 8 szt,15.52,6.01,72.09%,27.9%,1.872,8.0
4,2026-06-01,Futomaki TUŃCZYK SUROWY | 6 szt,2,45.0,90.0,Futomaki TUŃCZYK SUROWY | 6 szt,21.80,8.83,71.16%,28.8%,2.664,8.0
...,...,...,...,...,...,...,...,...,...,...,...,...
157,2026-06-07,Miso z łososiem | 350 ml,1,35.0,35.0,Miso z łososiem | 350 ml,18.77,3.59,83.95%,16.0%,1.944,8.0
158,2026-06-07,Futomaki FIRE HARMONY | 6 szt,1,58.0,58.0,Futomaki FIRE HARMONY | 6 szt,21.87,10.70,67.14%,32.9%,9.729,23.0
159,2026-06-07,Hosomaki AWOKADO | 8 szt,1,21.0,21.0,Hosomaki AWOKADO | 8 szt,13.49,4.73,74.04%,26.0%,1.584,8.0
160,2026-06-07,Futomaki ŁOSOŚ SUROWY | 6 szt,1,38.0,38.0,Futomaki ŁOSOŚ SUROWY | 6 szt,21.17,7.81,73.06%,26.9%,2.520,8.0


### 6.3. Review the integrated table

Remove the duplicate `sales_name` column. The resulting `sales_final` is the input for subsequent checks and analysis.

In [774]:
sales_final = sales_final.drop(columns='sales_name')

In [775]:
sales_final

,date,name,qty,price,sum,unit_profit,unit_cost,marginal_profit,food_cost,vat_as_cost,vat_rate
0,2026-06-01,Futomaki ŁOSOŚ GRILLOWANY | 6 szt,6,38.0,228.0,21.47,7.51,74.10%,25.9%,2.520,8.0
1,2026-06-01,Nigiri łosoś | 2 szt,4,25.0,100.0,15.88,5.65,73.77%,26.2%,1.872,8.0
2,2026-06-01,PROMO 1+1=3 Sushi burgery,3,120.0,360.0,64.00,35.36,64.42%,35.6%,8.640,8.0
3,2026-06-01,Hosomaki ŁOSOŚ SUROWY | 8 szt,3,25.0,75.0,15.52,6.01,72.09%,27.9%,1.872,8.0
4,2026-06-01,Futomaki TUŃCZYK SUROWY | 6 szt,2,45.0,90.0,21.80,8.83,71.16%,28.8%,2.664,8.0
...,...,...,...,...,...,...,...,...,...,...,...
157,2026-06-07,Miso z łososiem | 350 ml,1,35.0,35.0,18.77,3.59,83.95%,16.0%,1.944,8.0
158,2026-06-07,Futomaki FIRE HARMONY | 6 szt,1,58.0,58.0,21.87,10.70,67.14%,32.9%,9.729,23.0
159,2026-06-07,Hosomaki AWOKADO | 8 szt,1,21.0,21.0,13.49,4.73,74.04%,26.0%,1.584,8.0
160,2026-06-07,Futomaki ŁOSOŚ SUROWY | 6 szt,1,38.0,38.0,21.17,7.81,73.06%,26.9%,2.520,8.0


In [776]:
sales_final = sales_final[['date', 'name', 'qty', 'price', 'vat_rate', 'unit_cost', 'vat_as_cost', 'unit_profit', 'marginal_profit', 'food_cost', 'sum']]

In [777]:
sales_final = sales_final.rename({
    'price': 'unit_price',
    'food_cost': 'unit_food_cost',
    'sum': 'total_sum_of_sales_for_product'
}, axis=1)

In [778]:
sales_final

,date,name,qty,unit_price,vat_rate,unit_cost,vat_as_cost,unit_profit,marginal_profit,unit_food_cost,total_sum_of_sales_for_product
0,2026-06-01,Futomaki ŁOSOŚ GRILLOWANY | 6 szt,6,38.0,8.0,7.51,2.520,21.47,74.10%,25.9%,228.0
1,2026-06-01,Nigiri łosoś | 2 szt,4,25.0,8.0,5.65,1.872,15.88,73.77%,26.2%,100.0
2,2026-06-01,PROMO 1+1=3 Sushi burgery,3,120.0,8.0,35.36,8.640,64.00,64.42%,35.6%,360.0
3,2026-06-01,Hosomaki ŁOSOŚ SUROWY | 8 szt,3,25.0,8.0,6.01,1.872,15.52,72.09%,27.9%,75.0
4,2026-06-01,Futomaki TUŃCZYK SUROWY | 6 szt,2,45.0,8.0,8.83,2.664,21.80,71.16%,28.8%,90.0
...,...,...,...,...,...,...,...,...,...,...,...
157,2026-06-07,Miso z łososiem | 350 ml,1,35.0,8.0,3.59,1.944,18.77,83.95%,16.0%,35.0
158,2026-06-07,Futomaki FIRE HARMONY | 6 szt,1,58.0,23.0,10.70,9.729,21.87,67.14%,32.9%,58.0
159,2026-06-07,Hosomaki AWOKADO | 8 szt,1,21.0,8.0,4.73,1.584,13.49,74.04%,26.0%,21.0
160,2026-06-07,Futomaki ŁOSOŚ SUROWY | 6 szt,1,38.0,8.0,7.81,2.520,21.17,73.06%,26.9%,38.0


<a id="checks"></a>
## 7. Data checks and estimated costs

### 7.1. Identify missing reference profit

List distinct product names with missing `unit_profit`. Missing reference data does not imply zero profit.

In [779]:
sales_final[sales_final['unit_profit'].isna()]['name'].drop_duplicates()

11                        Samurai roll | 8 szt
67                                   Sos unagi
70                         Sos spicy majo 30ml
71                              Sos unagi 30ml
84                                    Pałeczki
100                         Fanta jagoda 0.33l
101    Zestaw Philadelphia + Krabs Burger -50%
102                          Sos teriyaki 30ml
144                          Sos śliwkowy 30ml
150             Vibe roll ŁOSOŚ SUROWY | 8 szt
156                            Sos sojowy 30ml
Name: name, dtype: str

### 7.2. Check arithmetic consistency

Inspect rows where `price × qty` differs from `sum`. Because price was derived from these same fields, this checks internal consistency rather than independently validating the source. Strict equality can also flag floating-point representation differences.

In [780]:
sales_final[(sales_final['unit_price'] * sales_final['qty'] != sales_final['total_sum_of_sales_for_product'])]

,date,name,qty,unit_price,vat_rate,unit_cost,vat_as_cost,unit_profit,marginal_profit,unit_food_cost,total_sum_of_sales_for_product


### 7.3. Estimate product costs for units sold

Multiply reference unit cost by quantity. `total_cost` estimates product costs using the available reference; it is not a net-profit calculation.

In [781]:
sales_final['total_cost'] = sales_final['unit_cost'] * sales_final['qty']

In [782]:
sales_final

,date,name,qty,unit_price,vat_rate,unit_cost,vat_as_cost,unit_profit,marginal_profit,unit_food_cost,total_sum_of_sales_for_product,total_cost
0,2026-06-01,Futomaki ŁOSOŚ GRILLOWANY | 6 szt,6,38.0,8.0,7.51,2.520,21.47,74.10%,25.9%,228.0,45.06
1,2026-06-01,Nigiri łosoś | 2 szt,4,25.0,8.0,5.65,1.872,15.88,73.77%,26.2%,100.0,22.60
2,2026-06-01,PROMO 1+1=3 Sushi burgery,3,120.0,8.0,35.36,8.640,64.00,64.42%,35.6%,360.0,106.08
3,2026-06-01,Hosomaki ŁOSOŚ SUROWY | 8 szt,3,25.0,8.0,6.01,1.872,15.52,72.09%,27.9%,75.0,18.03
4,2026-06-01,Futomaki TUŃCZYK SUROWY | 6 szt,2,45.0,8.0,8.83,2.664,21.80,71.16%,28.8%,90.0,17.66
...,...,...,...,...,...,...,...,...,...,...,...,...
157,2026-06-07,Miso z łososiem | 350 ml,1,35.0,8.0,3.59,1.944,18.77,83.95%,16.0%,35.0,3.59
158,2026-06-07,Futomaki FIRE HARMONY | 6 szt,1,58.0,23.0,10.70,9.729,21.87,67.14%,32.9%,58.0,10.70
159,2026-06-07,Hosomaki AWOKADO | 8 szt,1,21.0,8.0,4.73,1.584,13.49,74.04%,26.0%,21.0,4.73
160,2026-06-07,Futomaki ŁOSOŚ SUROWY | 6 szt,1,38.0,8.0,7.81,2.520,21.17,73.06%,26.9%,38.0,7.81


<a id="analysis"></a>
## 8. Revenue analysis

### 8.1. Aggregate sales by product

Sum quantity and revenue by `name` over the analysis period. The first cell also temporarily sums `price`; that intermediate value is not a meaningful unit price. The following cell replaces it with total product revenue divided by total product quantity.

In [783]:
sales_pivot = sales_final.groupby(by='name')[['qty', 'total_sum_of_sales_for_product', 'unit_price']].sum()
sales_pivot

,qty,total_sum_of_sales_for_product,unit_price
name,,,
Amour Set | 36 szt,1,250.0,250.0
Atlantic Wave roll | 8 szt,1,60.0,60.0
Black dragon roll | 8 szt,2,140.0,140.0
California KRAB-MIX | 8 szt,3,135.0,90.0
California KREWETKA GOTOWANA | 8 szt,2,110.0,110.0
...,...,...,...
Zestaw Philadelphia + Krabs Burger -50%,2,340.0,340.0
Zestaw Philadelphia | 24 szt,1,170.0,170.0
Zestaw Sakura | 62 szt,1,295.0,295.0


In [784]:
sales_pivot['price'] = sales_pivot['total_sum_of_sales_for_product'] / sales_pivot['qty']

### 8.2. Rank the top 10 products by revenue

Sort products by `sum` and display the ten largest contributors. This ranking measures revenue, not profit or units sold.

In [785]:
sales_pivot.sort_values(by='total_sum_of_sales_for_product', ascending=False).head(10)

,qty,total_sum_of_sales_for_product,unit_price,price
name,,,,
PROMO 1+1=3 Sushi burgery,25,3000.0,840.0,120.00
Futomaki ŁOSOŚ GRILLOWANY | 6 szt,50,1906.0,228.6,38.12
Futomaki FRESH ROLL | 6 szt,18,630.0,140.0,35.00
Zestaw Startowy | 16 szt,7,573.3,409.5,81.90
Futomaki KALMAR PANKO | 6 szt,14,448.0,64.0,32.00
Sushi burger ŁOSOŚ SUROWY,6,375.0,250.0,62.50
Zestaw Philadelphia + Krabs Burger -50%,2,340.0,340.0,170.00
Zestaw Sakura | 62 szt,1,295.0,295.0,295.00
Yakuza roll | 8 szt,4,292.0,292.0,73.00


### 8.3. Calculate total revenue

Sum revenue across the full product summary before selecting the top 10. Despite its name, `total_sales` stores revenue, not a count of sales.

In [786]:
total_sales = sales_pivot['total_sum_of_sales_for_product'].sum()
total_sales

np.float64(15508.8)

### 8.4. Calculate each product's revenue share

**Revenue share (%) = product revenue / total revenue × 100.** A value of 3.70 means that the product contributed 3.70% of revenue within the analysis scope. Shares across all products should sum to approximately 100%; retain unrounded values for subsequent calculations.

In [787]:
sales_pivot['revenue_share_pct'] = sales_pivot['total_sum_of_sales_for_product'] / total_sales * 100
sales_pivot

,qty,total_sum_of_sales_for_product,unit_price,price,revenue_share_pct
name,,,,,
Amour Set | 36 szt,1,250.0,250.0,250.0,1.611988
Atlantic Wave roll | 8 szt,1,60.0,60.0,60.0,0.386877
Black dragon roll | 8 szt,2,140.0,140.0,70.0,0.902713
California KRAB-MIX | 8 szt,3,135.0,90.0,45.0,0.870474
California KREWETKA GOTOWANA | 8 szt,2,110.0,110.0,55.0,0.709275
...,...,...,...,...,...
Zestaw Philadelphia + Krabs Burger -50%,2,340.0,340.0,170.0,2.192304
Zestaw Philadelphia | 24 szt,1,170.0,170.0,170.0,1.096152
Zestaw Sakura | 62 szt,1,295.0,295.0,295.0,1.902146


### 8.5. Measure top-10 revenue concentration

Sum the revenue shares of the ten leading products. This describes revenue concentration; it does not establish profitability or the incremental effectiveness of promotions.

In [788]:
sales_pivot.sort_values(by='revenue_share_pct', ascending=False).head(10)['revenue_share_pct'].sum()

np.float64(52.41733725368823)

<a id="next"></a>
## 9. Findings and next steps

For the current June 1–7, 2026 sample:

- Revenue within the defined product-sales scope totals **PLN 15,508.80**.
- The top **10 of 90 product names** contribute **52.42%** of that revenue; the remaining names contribute approximately **47.58%**.
- These findings describe the weekly revenue mix, not realized restaurant profit.


**The next steps is** clear up metrics data by updating formulas for calculating costs, profits, food costs

### 9.1. Create sales_metrics as a final data frame which contains complete information about unit-economics system in my product-data-base

unit profit measures how much we get profit by sell one selected product. It measures by **(product_price - product_cost - vat_as_cost)**.

product_cost contains costs of products only. These costs are calculated by an external table.

vat_as_cost measures by **(unit_price * vat_rate / (100 + vat_rate))**

Estimated contribution = sales revenue excluding VAT − gross ingredient cost. Commissions and other operating expenses are excluded

In [789]:
sales_metrics = sales_final[['date', 'name', 'qty', 'unit_price', 'vat_rate', 'unit_cost']]
sales_metrics

,date,name,qty,unit_price,vat_rate,unit_cost
0,2026-06-01,Futomaki ŁOSOŚ GRILLOWANY | 6 szt,6,38.0,8.0,7.51
1,2026-06-01,Nigiri łosoś | 2 szt,4,25.0,8.0,5.65
2,2026-06-01,PROMO 1+1=3 Sushi burgery,3,120.0,8.0,35.36
3,2026-06-01,Hosomaki ŁOSOŚ SUROWY | 8 szt,3,25.0,8.0,6.01
4,2026-06-01,Futomaki TUŃCZYK SUROWY | 6 szt,2,45.0,8.0,8.83
...,...,...,...,...,...,...
157,2026-06-07,Miso z łososiem | 350 ml,1,35.0,8.0,3.59
158,2026-06-07,Futomaki FIRE HARMONY | 6 szt,1,58.0,23.0,10.70
159,2026-06-07,Hosomaki AWOKADO | 8 szt,1,21.0,8.0,4.73
160,2026-06-07,Futomaki ŁOSOŚ SUROWY | 6 szt,1,38.0,8.0,7.81


In [790]:
sales_metrics['unit_vat'] = sales_metrics['unit_price'] * sales_metrics['vat_rate'] / (100 + sales_metrics['vat_rate'])

In [791]:
sales_metrics

,date,name,qty,unit_price,vat_rate,unit_cost,unit_vat
0,2026-06-01,Futomaki ŁOSOŚ GRILLOWANY | 6 szt,6,38.0,8.0,7.51,2.814815
1,2026-06-01,Nigiri łosoś | 2 szt,4,25.0,8.0,5.65,1.851852
2,2026-06-01,PROMO 1+1=3 Sushi burgery,3,120.0,8.0,35.36,8.888889
3,2026-06-01,Hosomaki ŁOSOŚ SUROWY | 8 szt,3,25.0,8.0,6.01,1.851852
4,2026-06-01,Futomaki TUŃCZYK SUROWY | 6 szt,2,45.0,8.0,8.83,3.333333
...,...,...,...,...,...,...,...
157,2026-06-07,Miso z łososiem | 350 ml,1,35.0,8.0,3.59,2.592593
158,2026-06-07,Futomaki FIRE HARMONY | 6 szt,1,58.0,23.0,10.70,10.845528
159,2026-06-07,Hosomaki AWOKADO | 8 szt,1,21.0,8.0,4.73,1.555556
160,2026-06-07,Futomaki ŁOSOŚ SUROWY | 6 szt,1,38.0,8.0,7.81,2.814815


In [792]:
sales_metrics = sales_metrics.rename({'unit_price': 'unit_price_gross'}, axis=1)
sales_metrics

,date,name,qty,unit_price_gross,vat_rate,unit_cost,unit_vat
0,2026-06-01,Futomaki ŁOSOŚ GRILLOWANY | 6 szt,6,38.0,8.0,7.51,2.814815
1,2026-06-01,Nigiri łosoś | 2 szt,4,25.0,8.0,5.65,1.851852
2,2026-06-01,PROMO 1+1=3 Sushi burgery,3,120.0,8.0,35.36,8.888889
3,2026-06-01,Hosomaki ŁOSOŚ SUROWY | 8 szt,3,25.0,8.0,6.01,1.851852
4,2026-06-01,Futomaki TUŃCZYK SUROWY | 6 szt,2,45.0,8.0,8.83,3.333333
...,...,...,...,...,...,...,...
157,2026-06-07,Miso z łososiem | 350 ml,1,35.0,8.0,3.59,2.592593
158,2026-06-07,Futomaki FIRE HARMONY | 6 szt,1,58.0,23.0,10.70,10.845528
159,2026-06-07,Hosomaki AWOKADO | 8 szt,1,21.0,8.0,4.73,1.555556
160,2026-06-07,Futomaki ŁOSOŚ SUROWY | 6 szt,1,38.0,8.0,7.81,2.814815


In [793]:
sales_metrics['unit_price_net'] = sales_metrics['unit_price_gross'] - sales_metrics['unit_vat']
sales_metrics['unit_profit_est'] = sales_metrics['unit_price_net'] - sales_metrics['unit_cost']
sales_metrics['margin_rate'] = sales_metrics['unit_profit_est'] / sales_metrics['unit_price_net']
sales_metrics['food_cost_rate'] = sales_metrics['unit_cost'] / sales_metrics['unit_price_net']
sales_metrics['total_sales_net'] = sales_metrics['qty'] * sales_metrics['unit_price_net']
sales_metrics['total_sales_gross'] = sales_metrics['qty'] * sales_metrics['unit_price_gross']
sales_metrics['total_cost_est'] = sales_metrics['qty'] * sales_metrics['unit_cost']
sales_metrics['total_profit_est'] = sales_metrics['total_sales_net'] - sales_metrics['total_cost_est']

sales_metrics

,date,name,qty,unit_price_gross,vat_rate,unit_cost,unit_vat,unit_price_net,unit_profit_est,margin_rate,food_cost_rate,total_sales_net,total_sales_gross,total_cost_est,total_profit_est
0,2026-06-01,Futomaki ŁOSOŚ GRILLOWANY | 6 szt,6,38.0,8.0,7.51,2.814815,35.185185,27.675185,0.786558,0.213442,211.111111,228.0,45.06,166.051111
1,2026-06-01,Nigiri łosoś | 2 szt,4,25.0,8.0,5.65,1.851852,23.148148,17.498148,0.755920,0.244080,92.592593,100.0,22.60,69.992593
2,2026-06-01,PROMO 1+1=3 Sushi burgery,3,120.0,8.0,35.36,8.888889,111.111111,75.751111,0.681760,0.318240,333.333333,360.0,106.08,227.253333
3,2026-06-01,Hosomaki ŁOSOŚ SUROWY | 8 szt,3,25.0,8.0,6.01,1.851852,23.148148,17.138148,0.740368,0.259632,69.444444,75.0,18.03,51.414444
4,2026-06-01,Futomaki TUŃCZYK SUROWY | 6 szt,2,45.0,8.0,8.83,3.333333,41.666667,32.836667,0.788080,0.211920,83.333333,90.0,17.66,65.673333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157,2026-06-07,Miso z łososiem | 350 ml,1,35.0,8.0,3.59,2.592593,32.407407,28.817407,0.889223,0.110777,32.407407,35.0,3.59,28.817407
158,2026-06-07,Futomaki FIRE HARMONY | 6 szt,1,58.0,23.0,10.70,10.845528,47.154472,36.454472,0.773086,0.226914,47.154472,58.0,10.70,36.454472
159,2026-06-07,Hosomaki AWOKADO | 8 szt,1,21.0,8.0,4.73,1.555556,19.444444,14.714444,0.756743,0.243257,19.444444,21.0,4.73,14.714444
160,2026-06-07,Futomaki ŁOSOŚ SUROWY | 6 szt,1,38.0,8.0,7.81,2.814815,35.185185,27.375185,0.778032,0.221968,35.185185,38.0,7.81,27.375185


In [794]:
sales_metrics.dtypes

date                 datetime64[us]
name                            str
qty                           int64
unit_price_gross            float64
vat_rate                    float64
unit_cost                   float64
unit_vat                    float64
unit_price_net              float64
unit_profit_est             float64
margin_rate                 float64
food_cost_rate              float64
total_sales_net             float64
total_sales_gross           float64
total_cost_est              float64
total_profit_est            float64
dtype: object